# Descriptive Analysis - Voting on Bonds 

In [2]:
import polars as pl 
from scipy import stats
data_dir = '~/Dropbox/Voting on Bonds/Data'
# load data 
data = pl.read_parquet(f'{data_dir}/Mergent/Clean/241113_issue_level_aggregation.gzip')
data = data.filter(pl.col('school').eq(0))
data = data.with_columns(GO = pl.when(pl.col('rev').eq(0)).then(1).otherwise(0))

## First, filter to bonds were vote requirement information is clear in state laws
vote_requirement = 1 when vote is clearly required for bond type \
vote_requirement = 0 when vote is clearly not required for bond type \
vote_requirement = Null when vote requirement is unclear for bond type 

In [3]:
print(data.group_by('vote_required').len())
data = data.filter(pl.col('vote_required').is_not_null())

shape: (3, 2)
┌───────────────┬───────┐
│ vote_required ┆ len   │
│ ---           ┆ ---   │
│ i32           ┆ u32   │
╞═══════════════╪═══════╡
│ null          ┆ 12029 │
│ 0             ┆ 12130 │
│ 1             ┆ 6856  │
└───────────────┴───────┘


## Counts of number of GO bonds and Revenue bonds by vote requirement

In [5]:
print(data
      .group_by(['vote_required', 'rev'])
      .agg(pl.col('issue_id').count().alias('num_issues'), 
           pl.col('seed_issuer').n_unique().alias('num_seed_issuers'))
      .sort(['vote_required', 'rev']))

shape: (4, 4)
┌───────────────┬─────┬────────────┬──────────────────┐
│ vote_required ┆ rev ┆ num_issues ┆ num_seed_issuers │
│ ---           ┆ --- ┆ ---        ┆ ---              │
│ i32           ┆ i8  ┆ u32        ┆ u32              │
╞═══════════════╪═════╪════════════╪══════════════════╡
│ 0             ┆ 0   ┆ 5339       ┆ 1439             │
│ 0             ┆ 1   ┆ 6791       ┆ 2460             │
│ 1             ┆ 0   ┆ 6327       ┆ 2050             │
│ 1             ┆ 1   ┆ 529        ┆ 202              │
└───────────────┴─────┴────────────┴──────────────────┘


## City Level - Counts of Rev and GO Bonds by State-Level Vote Requirement 
Here, only look at issuances where states laws are clear on both GO and Revenue 


In [47]:
(data
      .filter(pl.col('city').eq(1) & pl.col('City_GO_Vote').is_not_null() & pl.col('City_Rev_Vote').is_not_null())
      .group_by(['City_GO_Vote', 'City_Rev_Vote'])
      .agg(pl.col('state').n_unique().alias('num_states'), 
        pl.col('seed_issuer').n_unique().alias('num_seed_issuers'),
      pl.col('rev').sum().alias('revenue_bonds'),
           pl.col('GO').sum().alias('GO_bonds'))
      .with_columns(percent_rev = pl.col('revenue_bonds').truediv((pl.col('revenue_bonds').add(pl.col('GO_bonds'))))))

City_GO_Vote,City_Rev_Vote,num_states,num_seed_issuers,revenue_bonds,GO_bonds,percent_rev
i64,i64,u32,u32,i64,i32,f64
1,0,21,1837,2985,4195,0.415738
0,0,9,1227,927,3716,0.199655
1,1,6,221,417,262,0.614138


Composition of bonds seems to shift more toward GO when votes are not required 

## County-Level - Counts of Rev and GO Bonds by State-Level Vote Requirement 

In [14]:
print(data
      .filter(pl.col('county').eq(1) & pl.col('County_GO_Vote').is_not_null() & pl.col('County_Rev_Vote').is_not_null())
      .group_by(['County_GO_Vote', 'County_Rev_Vote'])
      .agg(pl.col('state').n_unique().alias('num_states'),
           pl.col('seed_issuer').n_unique().alias('num_seed_issuers'),
pl.col('rev').sum().alias('revenue_bonds'),
           pl.col('GO').sum().alias('GO_bonds'),)
      .with_columns(percent_rev = pl.col('revenue_bonds').truediv((pl.col('revenue_bonds').add(pl.col('GO_bonds'))))))

shape: (3, 7)
┌───────────────┬──────────────┬────────────┬──────────────┬──────────────┬──────────┬─────────────┐
│ County_GO_Vot ┆ County_Rev_V ┆ num_states ┆ num_seed_iss ┆ revenue_bond ┆ GO_bonds ┆ percent_rev │
│ e             ┆ ote          ┆ ---        ┆ uers         ┆ s            ┆ ---      ┆ ---         │
│ ---           ┆ ---          ┆ u32        ┆ ---          ┆ ---          ┆ i32      ┆ f64         │
│ i64           ┆ i64          ┆            ┆ u32          ┆ i64          ┆          ┆             │
╞═══════════════╪══════════════╪════════════╪══════════════╪══════════════╪══════════╪═════════════╡
│ 1             ┆ 0            ┆ 16         ┆ 612          ┆ 762          ┆ 1419     ┆ 0.349381    │
│ 0             ┆ 0            ┆ 10         ┆ 408          ┆ 279          ┆ 1623     ┆ 0.146688    │
│ 1             ┆ 1            ┆ 8          ┆ 67           ┆ 112          ┆ 77       ┆ 0.592593    │
└───────────────┴──────────────┴────────────┴──────────────┴──────────────┴──

Again, fraction of revenue bonds is much lower when GO vote is not required 

# Composition of GO Bond and Revenue Bond Sample 
* What fraction come from vote-requiring and non-vote-requiring states?

first just report % of city issuers in the sample (regardless of bond type) subject to both voting requirements 

In [44]:
data_filtered = (data
                 .filter(pl.col('city').eq(1))
                .filter(pl.col('City_GO_Vote').is_not_null())
                 )
go_vote = (data_filtered
           .group_by('City_GO_Vote')
           .agg(pl.col('seed_issuer').n_unique().alias('num_seed_issuers'))
           .with_columns(percent_issuers = pl.col('num_seed_issuers').truediv(data_filtered['seed_issuer'].n_unique())))

go_vote

City_GO_Vote,num_seed_issuers,percent_issuers
i64,u32,f64
0,1227,0.36334
1,2153,0.637548


In [45]:
data_filtered = (data
                 .filter(pl.col('city').eq(1))
                .filter(pl.col('City_Rev_Vote').is_not_null())
                 )
rev_vote = (data_filtered
           .group_by('City_Rev_Vote')
           .agg(pl.col('seed_issuer').n_unique().alias('num_seed_issuers'))
           .with_columns(percent_issuers = pl.col('num_seed_issuers').truediv(data_filtered['seed_issuer'].n_unique())))

rev_vote

City_Rev_Vote,num_seed_issuers,percent_issuers
i64,u32,f64
0,3738,0.944178
1,221,0.055822


## City-Level GO Bonds - Composition by GO Bond Vote Requirement

In [33]:
go = data.filter(pl.col('GO').eq(1)).filter(pl.col('city').eq(1)).filter(pl.col('City_GO_Vote').is_not_null())

go_agg = (go.group_by('City_GO_Vote')
          .agg(pl.col('issue_id').count().alias('num_issues'), 
               pl.col('seed_issuer').n_unique().alias('num_seed_issuers'))
          .with_columns(percent_issues = pl.col('num_issues').truediv(go.shape[0]), 
                        percent_issuers = pl.col('num_seed_issuers').truediv(go['seed_issuer'].n_unique())))
          
go_agg

City_GO_Vote,num_issues,num_seed_issuers,percent_issues,percent_issuers
i64,u32,u32,f64,f64
0,3716,1090,0.436406,0.4202
1,4799,1506,0.563594,0.580571


## City-Level Revenue Bonds - Composition by GO Bond Vote Requirement

In [46]:
rev = data.filter(pl.col('rev').eq(1)).filter(pl.col('city').eq(1)).filter(pl.col('City_GO_Vote').is_not_null())

rev_agg = (rev.group_by('City_GO_Vote')
          .agg(pl.col('issue_id').count().alias('num_issues'), 
               pl.col('seed_issuer').n_unique().alias('num_seed_issuers'))
          .with_columns(percent_issues = pl.col('num_issues').truediv(go.shape[0]), 
                        percent_issuers = pl.col('num_seed_issuers').truediv(go['seed_issuer'].n_unique())))
          
rev_agg

City_GO_Vote,num_issues,num_seed_issuers,percent_issues,percent_issuers
i64,u32,u32,f64,f64
0,927,350,0.214137,0.232868
1,3402,1153,0.785863,0.767132


## City-Level Revenue Bonds - Composition by Revenue Bond Vote Requirement

In [38]:
rev = data.filter(pl.col('rev').eq(1)).filter(pl.col('city').eq(1)).filter(pl.col('City_Rev_Vote').is_not_null())

rev_agg = (rev.group_by('City_Rev_Vote')
          .agg(pl.col('issue_id').count().alias('num_issues'), 
               pl.col('seed_issuer').n_unique().alias('num_seed_issuers'))
          .with_columns(percent_issues = pl.col('num_issues').truediv(go.shape[0]), 
                        percent_issuers = pl.col('num_seed_issuers').truediv(go['seed_issuer'].n_unique())))
          
rev_agg

City_Rev_Vote,num_issues,num_seed_issuers,percent_issues,percent_issuers
i64,u32,u32,f64,f64
0,5610,2012,1.295911,1.338656
1,417,168,0.096327,0.111776


# Univariate Analysis of Outcomes 

In [15]:
variables = ['weighted_avg_offering_yield', 'markup', 'markup_retail', 'markup_small_retail', 'markup_institutional',
          'markup_small_institutional', 'markup_large_institutional', 'yield_volatility']

In [18]:
# create function 
def univariate_analysis(variables, GO, city):
    outcomes = []
    for var in variables:
        # Filter the data frame based on the Treat variable
        vote = data.filter(pl.col("vote_required") == 1).filter(pl.col('city') == city).filter(pl.col('GO') == GO)
        no_vote = data.filter(pl.col("vote_required") == 0).filter(pl.col('city') == city).filter(pl.col('GO') == GO)
        
        # Calculate the mean value for each group for the current variable
        mean_vote = vote.select(pl.col(var)).mean()[var][0]
        mean_no_vote = no_vote.select(pl.col(var)).mean()[var][0]
        
        # Calculate the t-statistic for the current variable
        t_statistic, p_value = stats.ttest_ind(vote.filter(pl.col(var).is_not_null()).select(pl.col(var)).to_numpy(), 
                                               no_vote.filter(pl.col(var).is_not_null()).select(pl.col(var)).to_numpy())
        
        output = {'variable': var, 
                            'Vote Required': mean_vote, 
                            'Vote Required Count': vote.shape[0],
                            'No Vote Required': mean_no_vote,
                            'No Vote Required Count': no_vote.shape[0],
                            'Difference': mean_vote - mean_no_vote, 
                            't-statistic': t_statistic, 
                            'p-value':p_value}
        
        outcomes.append(output)
    
    return pl.DataFrame(outcomes)

### City-Level - GO Bonds 

In [19]:
df = univariate_analysis(variables, city = 1, GO = 1)
df

variable,Vote Required,Vote Required Count,No Vote Required,No Vote Required Count,Difference,t-statistic,p-value
str,f64,i64,f64,i64,f64,f64,f64
"""weighted_avg_offering_yield""",3.134833,4799,3.058785,3716,0.076048,3.118436,0.001824
"""markup""",65.445175,4799,65.439106,3716,0.006069,0.006076,0.995152
"""markup_retail""",70.901336,4799,70.746748,3716,0.154588,0.13006,0.896524
"""markup_small_retail""",81.202124,4799,79.242018,3716,1.960106,1.27934,0.200844
"""markup_institutional""",53.3387,4799,52.788969,3716,0.549731,0.589844,0.555321
"""markup_small_institutional""",55.216789,4799,54.33248,3716,0.884309,0.846328,0.397412
"""markup_large_institutional""",47.005245,4799,48.595767,3716,-1.590522,-1.017846,0.308816
"""yield_volatility""",0.67778,4799,0.674608,3716,0.003172,0.370755,0.710834


### County-Level - GO Bonds

In [20]:
df = univariate_analysis(variables, city = 0, GO = 1)
df

variable,Vote Required,Vote Required Count,No Vote Required,No Vote Required Count,Difference,t-statistic,p-value
str,f64,i64,f64,i64,f64,f64,f64
"""weighted_avg_offering_yield""",3.144601,1528,2.938181,1623,0.20642,4.992841,6.2733e-7
"""markup""",64.973604,1528,70.704918,1623,-5.731315,-3.505184,0.000465
"""markup_retail""",72.7173,1528,76.896194,1623,-4.178893,-2.159018,0.030958
"""markup_small_retail""",82.464182,1528,83.687078,1623,-1.222896,-0.527747,0.597736
"""markup_institutional""",49.839106,1528,57.161451,1623,-7.322345,-4.985696,6.6612e-7
"""markup_small_institutional""",53.187983,1528,60.540516,1623,-7.352533,-4.35137,0.000014
"""markup_large_institutional""",40.525068,1528,50.084409,1623,-9.559341,-6.076686,1.4970e-9
"""yield_volatility""",0.705239,1528,0.669039,1623,0.0362,2.962516,0.003082


### City Level - Revenue Bonds

In [21]:
df = univariate_analysis(variables, city = 1, GO = 0)
df

variable,Vote Required,Vote Required Count,No Vote Required,No Vote Required Count,Difference,t-statistic,p-value
str,f64,i64,f64,i64,f64,f64,f64
"""weighted_avg_offering_yield""",3.484069,417,3.915174,5610,-0.431105,-5.970705,2.4970e-9
"""markup""",76.104919,417,89.703064,5610,-13.598145,-4.29154,0.000018
"""markup_retail""",82.312625,417,98.765984,5610,-16.453359,-4.596841,0.000004
"""markup_small_retail""",93.257911,417,110.150746,5610,-16.892835,-3.952107,0.000079
"""markup_institutional""",56.563649,417,66.991438,5610,-10.427789,-3.791652,0.000152
"""markup_small_institutional""",60.036429,417,73.524486,5610,-13.488058,-4.172248,0.000031
"""markup_large_institutional""",48.395262,417,53.31202,5610,-4.916758,-1.653872,0.098267
"""yield_volatility""",0.710292,417,0.681518,5610,0.028775,0.991496,0.321505


### County-Level-Revenue Bonds

In [22]:
df = univariate_analysis(variables, city = 0, GO = 0)
df

variable,Vote Required,Vote Required Count,No Vote Required,No Vote Required Count,Difference,t-statistic,p-value
str,f64,i64,f64,i64,f64,f64,f64
"""weighted_avg_offering_yield""",4.005006,112,3.947407,1181,0.0576,0.397773,0.690863
"""markup""",75.876052,112,91.871191,1181,-15.995139,-2.569654,0.010333
"""markup_retail""",85.473343,112,102.370352,1181,-16.89701,-2.469484,0.013717
"""markup_small_retail""",98.772888,112,114.395105,1181,-15.622218,-1.985893,0.047381
"""markup_institutional""",52.381661,112,67.798478,1181,-15.416817,-2.940707,0.003361
"""markup_small_institutional""",64.423841,112,78.859408,1181,-14.435567,-2.297945,0.02181
"""markup_large_institutional""",34.877918,112,49.446381,1181,-14.568463,-3.055583,0.002325
"""yield_volatility""",0.725974,112,0.709381,1181,0.016593,0.40217,0.687651


# Univariate Analysis of Characteristics 

In [23]:
variables = ['log_issue_size', 'log_weighted_avg_maturity', 'ln_num_cusip', 'weighted_avg_callable', 'weighted_avg_insured',
 'weighted_avg_sinkable', 'pop','gdp', 'pers_inc', 'percap_inc']

### City-Level - GO Bonds

In [24]:
df = univariate_analysis(variables, city = 1, GO = 1)
df

variable,Vote Required,Vote Required Count,No Vote Required,No Vote Required Count,Difference,t-statistic,p-value
str,f64,i64,f64,i64,f64,f64,f64
"""log_issue_size""",15.170353,4799,15.24321,3716,-0.072857,-2.359632,0.018316
"""log_weighted_avg_maturity""",2.223954,4799,2.151101,3716,0.072853,7.145785,9.6861e-13
"""ln_num_cusip""",2.578643,4799,2.68666,3716,-0.108017,-9.243125,2.9769e-20
"""weighted_avg_callable""",0.536117,4799,0.438148,3716,0.097969,16.655015,2.5820e-61
"""weighted_avg_insured""",0.244712,4799,0.329221,3716,-0.084509,-8.68456,4.5087e-18
"""weighted_avg_sinkable""",0.166703,4799,0.106748,3716,0.059955,10.007459,1.9042e-23
"""pop""",507729.303234,4799,466194.539584,3716,41534.76365,2.420642,0.015517
"""gdp""",3.1832e7,4799,3.1475e7,3716,356928.184705,0.302918,0.761961
"""pers_inc""",2.1681e7,4799,2.4265e7,3716,-2.5833e6,-3.251355,0.001154


### County-Level - GO Bonds

In [25]:
df = univariate_analysis(variables, city = 0, GO = 1)
df

variable,Vote Required,Vote Required Count,No Vote Required,No Vote Required Count,Difference,t-statistic,p-value
str,f64,i64,f64,i64,f64,f64,f64
"""log_issue_size""",15.833837,1528,15.834934,1623,-0.001097,-0.020314,0.983794
"""log_weighted_avg_maturity""",2.181831,1528,2.175806,1623,0.006025,0.320439,0.748657
"""ln_num_cusip""",2.574915,1528,2.717157,1623,-0.142242,-7.383124,1.9690e-13
"""weighted_avg_callable""",0.498162,1528,0.466689,1623,0.031473,2.985911,0.002849
"""weighted_avg_insured""",0.262631,1528,0.321903,1623,-0.059272,-3.69127,0.000227
"""weighted_avg_sinkable""",0.112335,1528,0.088856,1623,0.023479,2.807358,0.005026
"""pop""",304231.673049,1528,326479.175573,1623,-22247.502524,-1.206872,0.227584
"""gdp""",2.0077e7,1528,1.9968e7,1623,109103.23486,0.081078,0.935386
"""pers_inc""",1.3852e7,1528,1.6443e7,1623,-2.5910e6,-2.682414,0.007352


### City-Level - Revenue Bonds

In [26]:
df = univariate_analysis(variables, city = 1, GO = 0)
df

variable,Vote Required,Vote Required Count,No Vote Required,No Vote Required Count,Difference,t-statistic,p-value
str,f64,i64,f64,i64,f64,f64,f64
"""log_issue_size""",15.408244,417,15.720986,5610,-0.312742,-3.885346,0.000103
"""log_weighted_avg_maturity""",2.262154,417,NaN,5610,NaN,NaN,NaN
"""ln_num_cusip""",2.255589,417,2.413809,5610,-0.15822,-3.931708,0.000085
"""weighted_avg_callable""",0.684963,417,0.69589,5610,-0.010927,-0.814466,0.41541
"""weighted_avg_insured""",0.308564,417,0.315967,5610,-0.007403,-0.315343,0.752512
"""weighted_avg_sinkable""",0.349492,417,0.366587,5610,-0.017095,-0.866485,0.386259
"""pop""",174092.078261,417,794110.880631,5610,-620018.80237,-7.037714,2.2061e-12
"""gdp""",1.0002e7,417,5.1131e7,5610,-4.1128e7,-7.120198,1.2236e-12
"""pers_inc""",7.7258e6,417,3.5086e7,5610,-2.7360e7,-6.853428,8.0430e-12


### County-Level - Revenue Bonds

In [27]:
df = univariate_analysis(variables, city = 0, GO = 0)
df

variable,Vote Required,Vote Required Count,No Vote Required,No Vote Required Count,Difference,t-statistic,p-value
str,f64,i64,f64,i64,f64,f64,f64
"""log_issue_size""",16.69988,112,16.512029,1181,0.187851,1.213109,0.22531
"""log_weighted_avg_maturity""",2.517094,112,NaN,1181,NaN,NaN,NaN
"""ln_num_cusip""",2.262654,112,2.367645,1181,-0.104991,-1.231913,0.218206
"""weighted_avg_callable""",0.631819,112,0.679307,1181,-0.047488,-1.530876,0.126045
"""weighted_avg_insured""",0.35607,112,0.290128,1181,0.065942,1.47655,0.14004
"""weighted_avg_sinkable""",0.307296,112,0.364914,1181,-0.057618,-1.524445,0.127643
"""pop""",555181.587629,112,516763.266793,1181,38418.320836,0.490056,0.624188
"""gdp""",3.4310e7,112,3.1919e7,1181,2.3912e6,0.446687,0.655185
"""pers_inc""",2.4650e7,112,2.2559e7,1181,2.0905e6,0.554325,0.579464
